# DSS v1.0 — Pourquoi, quel impact, et si, quelle stratégie

## Objectif

La v0.1 du moteur (`03_decision_engine.ipynb`) répond à *"quel est le risque ?"*. Un vrai système d'aide à la décision doit aussi répondre à :

- **Pourquoi** ce risque est-il là ? (causes)
- **Quel impact** aurait une collecte supplémentaire ? (chiffré)
- **Et si** on change une hypothèse — combien collecter, combien transférer, demande en hausse ? (simulation)
- **Quelle stratégie** choisir entre plusieurs options ? (comparaison)

Ces 4 fonctions vivent dans src/decision_engine.py, au même titre que le moteur de risque de la v0.1. Toujours des règles explicites — pas de modèle prédictif tant qu'on n'a pas de données réelles pour le calibrer (ça, c'est la v2.0).

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd()))
from src import decision_engine as de

df = pd.read_csv(Path.cwd() / "cnts_simulated_blood_demand.csv")
latest_month = df["month"].max()

snapshot, transfers = de.build_recommendations(df, latest_month)
snapshot = de.explain_causes(snapshot, df, latest_month)

top = snapshot.iloc[0]
top[["region", "blood_group", "risk_level_calc", "coverage_ratio_calc"]]

region                 Bouaké
blood_group                O+
risk_level_calc         Eleve
coverage_ratio_calc      0.68
Name: 0, dtype: object

## 1. Pourquoi ce risque ? (Évolution 1)

Les causes comparent le mois courant à la moyenne glissante des 3 mois précédents pour la même région x groupe sanguin — baisse de collecte, hausse de demande, ou simplement un stock sous le seuil de sécurité.

In [2]:
for cause in top["causes"]:
    print("-", cause)

- Hausse inhabituelle de la demande : 243 unites contre 191 en moyenne sur les 3 derniers mois (27% de plus).
- Stock sous le seuil de securite : couverture de 0.68 contre 0.85 minimum recommande.


## 2. Quel impact aurait une collecte ? (Évolution 2)

`units_needed_for_target` calcule le nombre d'unités à collecter pour changer de catégorie de risque — pas une estimation vague, un chiffre défendable devant un directeur.

In [3]:
impact = de.impact_of_collecte(top, target_risk="Moyen")
print(impact["text"])

Une collecte d'environ 43 unites supplementaires ferait passer Bouaké / O+ de Eleve a Moyen.


## 3. Et si ? (Évolution 3)

`simulate_scenario` recalcule stock, couverture et risque sous une hypothèse — sans toucher aux données sources. On vérifie ici que collecter le nombre d'unités recommandé à l'étape 2 fait bien basculer le risque, et qu'une hausse de demande dégrade la situation.

In [4]:
sim_collecte = de.simulate_scenario(top, collecte_delta=impact["units"])
print(f"Collecte de {impact['units']} unites : {top['risk_level_calc']} -> {sim_collecte['risk_level_scenario']} "
      f"(couverture {top['coverage_ratio_calc']:.2f} -> {sim_collecte['coverage_ratio_scenario']:.2f})")

sim_demande = de.simulate_scenario(top, demand_pct_change=0.15)
print(f"Demande +15% sans rien faire : {top['risk_level_calc']} -> {sim_demande['risk_level_scenario']} "
      f"(couverture {top['coverage_ratio_calc']:.2f} -> {sim_demande['coverage_ratio_scenario']:.2f})")

Collecte de 43 unites : Eleve -> Moyen (couverture 0.68 -> 0.86)
Demande +15% sans rien faire : Eleve -> Eleve (couverture 0.68 -> 0.59)


## 4. Comparer les stratégies (Évolution 4)

Sur le dernier mois, aucune région n'a de surplus mobilisable (voir `03_decision_engine.ipynb`) : la comparaison se limite à "collecte" vs "aucune action". Pour illustrer les 3 stratégies, on reprend l'exemple de juin 2024 (Bouaké / AB+), où un transfert est possible.

In [5]:
print("Comparaison sur la situation la plus critique du mois courant :")
display(de.compare_strategies(top, transfers))

snap_june, transfers_june = de.build_recommendations(df, "2024-06")
row_bouake_abplus = snap_june[(snap_june["region"] == "Bouaké") & (snap_june["blood_group"] == "AB+")].iloc[0]

print("\nComparaison sur Bouaké / AB+ (juin 2024, transfert disponible) :")
de.compare_strategies(row_bouake_abplus, transfers_june)

Comparaison sur la situation la plus critique du mois courant :


,strategie,detail,risque_resultant,cout_effort
0,Campagne de collecte,+43 unites collectees,Moyen,Moyen
1,Aucune action,Statu quo,Eleve,Risque de rupture eleve



Comparaison sur Bouaké / AB+ (juin 2024, transfert disponible) :


,strategie,detail,risque_resultant,cout_effort
0,Transfert,5 unites depuis Abidjan,Moyen,Faible
1,Campagne de collecte,+1 unites collectees,Moyen,Moyen
2,Aucune action,Statu quo,Eleve,Risque de rupture eleve


## Ce que ça démontre

- Chaque recommandation est maintenant accompagnée d'un **pourquoi** (causes), d'un **combien** (impact chiffré), d'un **et si** (simulation) et d'un **face à quoi** (comparaison de stratégies) — les 4 évolutions de la v1.0.
- Toujours zéro modèle prédictif : uniquement des règles arithmétiques sur les données déjà présentes. La v2.0 (prévision à 30 jours, calibrée sur données réelles) reste une étape distincte, volontairement pas construite maintenant.
- Ces fonctions sont directement branchées dans `app.py`, sous une nouvelle section "Analyse décisionnelle".